# Graph features

Demonstrates the graph-derived transaction network features (in/out degree, edge frequency, destination historical fraud rate), computed with plain DataFrame aggregations rather than a graph library.

**Prerequisites:** `make sample-data` and `make ingest`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
from pyspark.sql import functions as F

from transaction_risk.features.graph_features import add_graph_features
from transaction_risk.spark.io import read_table

transactions = read_table(spark, '../data/silver/transactions')
with_graph = add_graph_features(transactions)

with_graph.select(
    'nameOrig',
    'nameDest',
    'origin_out_degree',
    'destination_in_degree',
    'edge_frequency',
    'destination_historical_fraud_rate',
).show(10)

+--------+--------+-----------------+---------------------+--------------+---------------------------------+
|nameOrig|nameDest|origin_out_degree|destination_in_degree|edge_frequency|destination_historical_fraud_rate|
+--------+--------+-----------------+---------------------+--------------+---------------------------------+
|C0001028|C0001163|                4|                    3|             1|                              0.0|
|C0000271|C0000410|                5|                    2|             1|                              0.0|
|C0000024|C0001003|                6|                    2|             1|                              0.0|
|C0001075|C0000200|                8|                    2|             1|                              0.0|
|C0001162|M0000155|                4|                    2|             1|                              0.0|
|C0000930|C0000840|                4|                    3|             1|                              0.0|
|C0000772|C0001572|

In [3]:
# Destinations that concentrate incoming flows from many distinct accounts are mule-account candidates
(
    with_graph.select('nameDest', 'destination_in_degree', 'destination_historical_fraud_rate')
    .distinct()
    .orderBy(F.col('destination_in_degree').desc())
    .show(10)
)
spark.stop()

+--------+---------------------+---------------------------------+
|nameDest|destination_in_degree|destination_historical_fraud_rate|
+--------+---------------------+---------------------------------+
|C0000377|                    7|                              0.0|
|C0000642|                    7|                              0.0|
|M0001428|                    6|                              0.0|
|C0001667|                    6|                              0.0|
|M0001443|                    6|                              0.0|
|C0001274|                    6|                              0.0|
|C0001347|                    6|               0.3333333333333333|
|C0001014|                    6|                              0.0|
|C0000948|                    6|                              0.0|
|C0001040|                    6|                              0.0|
+--------+---------------------+---------------------------------+
only showing top 10 rows
